# Spam VII: Small language models as feature extractors

The last rung of the ladder. A **small language model (SLM)** is a Transformer with up to ~1B parameters, small enough to run on
a laptop CPU. Pre-trained on huge text collections, it has already learnt *what words and sentences mean*. We keep it **frozen**,
feed it every SMS and read out one dense vector (an **embedding**); the classifier on top can then be tiny (a *linear probe*).

| model | type | parameters | dimension |
|:--|:--|--:|--:|
| `all-MiniLM-L6-v2` | encoder (BERT-style), sentence-transformers | 22 M | 384 |
| `Qwen3-Embedding-0.6B` | decoder LLM turned into an embedder, instruction-aware | 600 M | 1024 |

(Alternatives with the same API: `google/embeddinggemma-300m`, which needs accepting the Gemma licence on Hugging Face,
`BAAI/bge-small-en-v1.5`, `intfloat/multilingual-e5-small`.)

**Requirements:** the optional dependency group `slm` (`pip install --group slm --extra-index-url https://download.pytorch.org/whl/cpu`).
The first run downloads the models (~90 MB and ~1.2 GB) and computes the embeddings once (a few minutes on CPU); results are cached in
`.cache/`. Set the environment variable `AAS_SLM=small` to use only MiniLM.

In [ ]:
import hashlib
import os
import time
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "jax")

import keras
import numpy as np
from matplotlib import pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import spamlib as sl

plt.rcParams["figure.dpi"] = 100
CACHE = Path(".cache")
CACHE.mkdir(exist_ok=True)

texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)
print(len(X_train), len(X_test))

## 1. Embedding function with a disk cache

Encoding is the expensive step (a forward pass of the whole network per message), so it is done **once** and cached.
Qwen3-Embedding is *instruction-aware*: a one-line task description in front of the text steers the embedding towards the task.

In [ ]:
MODELS = {
    "MiniLM (22M)": {"name": "sentence-transformers/all-MiniLM-L6-v2", "prompt": None},
    "Qwen3-Emb (0.6B)": {
        "name": "Qwen/Qwen3-Embedding-0.6B",
        "prompt": "Instruct: Classify whether this SMS message is spam or a legitimate message\nQuery: ",
    },
}
if os.environ.get("AAS_SLM") == "small":
    MODELS = {k: v for k, v in MODELS.items() if "MiniLM" in k}

_loaded = {}


def get_model(label):
    if label not in _loaded:
        m = SentenceTransformer(MODELS[label]["name"], device="cpu")
        m.max_seq_length = 128  # SMS are short; this bounds the cost
        _loaded[label] = m
    return _loaded[label]


def embed(label, docs):
    """L2-normalised embeddings, cached on disk (key = model + content of the documents)."""
    key = hashlib.sha1((MODELS[label]["name"] + "\n".join(docs)).encode()).hexdigest()[:16]
    path = CACHE / f"{key}.npy"
    if path.exists():
        return np.load(path)
    t0 = time.time()
    e = get_model(label).encode(
        docs, batch_size=64, prompt=MODELS[label]["prompt"], normalize_embeddings=True, show_progress_bar=False
    )
    print(f"  {label}: encoded {len(docs)} messages in {time.time() - t0:.0f}s")
    np.save(path, e)
    return e


emb = {label: (embed(label, X_train), embed(label, X_test)) for label in MODELS}
for label, (a, _) in emb.items():
    print(label, a.shape)

## 2. Semantic neighbours

The cosine similarity of two normalised vectors is their dot product. Messages that *mean* the same are close, even without a
common word (TF-IDF would score ~0 for the pair below).

In [ ]:
probe = [
    "You have won a free holiday, claim your cash prize now",
    "Congratulations! Collect your complimentary money reward",
]
for label in MODELS:
    m = get_model(label)
    pv = m.encode(probe, prompt=MODELS[label]["prompt"], normalize_embeddings=True)
    print(f"{label:18} cosine(probe1, probe2) = {float(pv[0] @ pv[1]):.3f}")

tfidf_pair = TfidfVectorizer().fit_transform(probe)
print(f"{'TF-IDF':18} cosine(probe1, probe2) = {float((tfidf_pair[0] @ tfidf_pair[1].T).toarray()[0, 0]):.3f}")

## 3. Linear probe: logistic regression on frozen embeddings

Embeddings from an LLM are **anisotropic** (all cosines are high, see above), so we *standardise* each dimension before the
logistic regression, and choose the regularisation strength $C$ by 3-fold cross-validation **on the training set**.

In [ ]:
def probe(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))


best_C = {}
results = {}
for label, (a_tr, a_te) in emb.items():
    cv_fit = GridSearchCV(
        probe(1.0), {"logisticregression__C": [1e-3, 1e-2, 1e-1, 1.0]}, cv=3, scoring="f1", n_jobs=-1
    ).fit(a_tr, y_train)
    best_C[label] = cv_fit.best_params_["logisticregression__C"]
    s = cv_fit.decision_function(a_te)
    results[f"{label} + LR"] = sl.scores(y_test, (s >= 0).astype(int), s)
print("C chosen by CV:", best_C)

tfidf_lr = make_pipeline(
    TfidfVectorizer(
        tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
    ),
    LogisticRegression(C=30, max_iter=2000),
).fit(X_train, y_train)
s = tfidf_lr.decision_function(X_test)
results["TF-IDF words + LR"] = sl.scores(y_test, (s >= 0).astype(int), s)
sl.show(results)

With plenty of labelled data the gap to TF-IDF is small on this corpus (the 2005-era British SMS are lexically very distinctive).
The advantage of pre-trained embeddings shows up when **labels are scarce**, exactly the situation of a *new* spam campaign.

## 4. Learning curves: how many labels do we need?

Train on random stratified subsets of $n$ messages (5 repetitions), test on the same held-out set.

In [ ]:
sizes = [20, 50, 100, 250, 500, 1000, len(X_train)]
curve = {"TF-IDF words + LR": [], **{f"{label} + LR": [] for label in MODELS}}
for n in sizes:
    reps = {k: [] for k in curve}
    splits = (
        [np.arange(len(X_train))]
        if n == len(X_train)
        else [
            idx
            for idx, _ in StratifiedShuffleSplit(n_splits=5, train_size=n, random_state=sl.SEED).split(
                np.zeros(len(y_train)), y_train
            )
        ]
    )
    for idx in splits:
        yt = y_train[idx]
        m = make_pipeline(
            TfidfVectorizer(
                tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), sublinear_tf=True
            ),
            LogisticRegression(C=30, max_iter=2000),
        ).fit([X_train[i] for i in idx], yt)
        reps["TF-IDF words + LR"].append(sl.scores(y_test, m.predict(X_test))["f1"])
        for label, (a_tr, a_te) in emb.items():
            c = probe(best_C[label]).fit(a_tr[idx], yt)
            reps[f"{label} + LR"].append(sl.scores(y_test, c.predict(a_te))["f1"])
    for k, v in reps.items():
        curve[k].append((np.mean(v), np.std(v)))

fig, ax = plt.subplots(figsize=(6.5, 4))
for k, v in curve.items():
    mean = np.array([a for a, _ in v])
    std = np.array([b for _, b in v])
    ax.plot(sizes, mean, "o-", label=k)
    ax.fill_between(sizes, mean - std, mean + std, alpha=0.15)
ax.set(xscale="log", xlabel="labelled training messages", ylabel="F1 (spam)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'labelled messages':22}" + "".join(f"{n:>8}" for n in sizes))
for k, v in curve.items():
    print(f"{k:22}" + "".join(f"{m:8.3f}" for m, _ in v))

## 5. A small neural head in Keras

A linear probe cannot bend the decision boundary; a tiny MLP can. We use **Keras 3 on the JAX backend** (no TensorFlow).
Validation data early-stops the training, and the test set is used once.

In [ ]:
label = list(MODELS)[-1]
a_tr, a_te = emb[label]
keras.utils.set_random_seed(sl.SEED)
norm = keras.layers.Normalization()
norm.adapt(a_tr)  # standardise with *training* statistics
head = keras.Sequential(
    [
        keras.layers.Input(shape=(a_tr.shape[1],)),
        norm,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)
head.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
hist = head.fit(
    a_tr,
    y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=64,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)
p = head.predict(a_te, verbose=0).ravel()
print(f"{label} + Keras MLP head: stopped after {len(hist.history['loss'])} epochs")
sl.show({f"{label} + MLP head": sl.scores(y_test, (p >= 0.5).astype(int), p)})

## 6. What does the space look like?

In [ ]:
a_tr, a_te = emb[list(MODELS)[0]]
xy = PCA(n_components=2, random_state=sl.SEED).fit(a_tr).transform(a_te)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for cls, name, color in ((0, "ham", "tab:blue"), (1, "spam", "tab:red")):
    ax.scatter(*xy[y_test == cls].T, s=6, alpha=0.5, label=name, color=color)
ax.legend()
ax.set_title(f"{list(MODELS)[0]}: PCA of test embeddings")
plt.tight_layout()
plt.show()

## 7. The price of meaning: cost per message

A better representation is not free. Latency and memory decide whether the model can sit on a mail gateway.

In [ ]:
sample = X_test[:200]
rows = []
t0 = time.time()
tfidf_lr.predict(sample)
rows.append(("TF-IDF + LR", (time.time() - t0) / len(sample) * 1000, "~1 MB"))
for label in MODELS:
    m = get_model(label)
    t0 = time.time()
    m.encode(sample, prompt=MODELS[label]["prompt"], batch_size=32, show_progress_bar=False)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    rows.append((label, (time.time() - t0) / len(sample) * 1000, f"{params:.0f}M params"))
print(f"{'model':20}{'ms / message':>14}   size")
for name, ms, size in rows:
    print(f"{name:20}{ms:14.2f}   {size}")

**Design rule.** Start with TF-IDF + a linear model; keep an SLM embedder as the *second stage* for messages the cheap model is
unsure about, or when labels are scarce. Cost matters for the defender; for the attacker, the embedder is a *differentiable,
downloadable* model, which makes white-box attacks possible (notebook 07).

## Exercises

1. Remove the instruction prompt from Qwen3-Embedding. Does the F1 change? What does that say about *prompt sensitivity*?
2. Compute the similarity of a ham message and its *paraphrase written by you*. Which representation (TF-IDF, fastText, SLM) keeps
   them closest?
3. Replace the MLP head by a linear SVM. Is the extra non-linearity worth its cost?
4. Use only the first 64 dimensions of the Qwen embedding (*Matryoshka* truncation, then re-normalise). How does the score degrade?